In [1]:
import logging
import warnings
import torch
import torchvision.transforms as transforms
import importlib
from components import FL_sim
from components.other_utilities import models_to_train
import components.broadcast_components.broadcasting_process.broadcast_reporting_utilities
import components.broadcast_components.broadcasting_process.ServerTrainingPerRoundProtocol
import components.broadcast_components.compressor.rans_coding
import components.other_utilities.datasets

importlib.reload(components.broadcast_components.compressor.rans_coding)
importlib.reload(components.broadcast_components.broadcasting_process.ServerTrainingPerRoundProtocol)
importlib.reload(components.broadcast_components.WZ_models.wz_quant_RNN)
importlib.reload(components.broadcast_components.broadcasting_process.broadcast_reporting_utilities)
importlib.reload(FL_sim)
importlib.reload(models_to_train)

from components.other_utilities.models_to_train import ResNetPLModel
from components.FL_sim import FLSimulator
from components.other_utilities.datasets import FasterSVHN
from components.broadcast_components.broadcasting_process.ServerTrainingPerRoundProtocol import WZServerTrainingPerRoundProtocol
from components.broadcast_components.broadcasting_process.broadcast_reporting_utilities import BroadcastMetricGatheringUtilities
from components.broadcast_components.WZ_models.wz_quant_RNN import PL_EncoderDecoder_RNN

torch.set_float32_matmul_precision('high')
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", "LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]")
warnings.filterwarnings("ignore", "You defined a `validation_step` but")
warnings.filterwarnings("ignore", "Starting from v1.9.0, `tensorboardX` has been")
warnings.filterwarnings("ignore", "The number of training batches")
warnings.filterwarnings("ignore", "`Trainer.fit` stopped: ")

In [2]:
dataset = [
    FasterSVHN(
        limit_count=1_500,
        root='./data/SVHN', split=s,
        transform=transforms.Compose([
            transforms.Resize(32),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.4377, 0.4438, 0.4728],
                std=[0.1980, 0.2010, 0.1970]
            ),
        ])
    ) for s in ['train', 'test']]

for i in range(10):
    for d in dataset:
        d.labels[i]=i

In [3]:
from components.broadcast_components.WZ_models.WZQuantizerWithDataPrep import QuantizerWithDataPrep

worker_count = 2
batch_size = 500

# *****************

user_logger = None#UnifiedLoggingClass(worker_count)
# ****
wz_model = PL_EncoderDecoder_RNN(inp_dim=1, side_info_size=0, num_planes=2,
                                 bins_per_plane=16, lr=1e-3, marginal=True).to(torch.float32)
path_to_basic = r'data/basicRNN_2plane_4bins_state.pt'
wz_model.load_state_dict(torch.load(path_to_basic, map_location='cpu'))
# ****
base_quantizer = QuantizerWithDataPrep(wz_model, train_sample_size=100_000,
        count_side_info_data=0, enable_progress_bar=False, user_logger=user_logger)
broadcast_prot_base = WZServerTrainingPerRoundProtocol(worker_count, base_quantizer, no_global_quantization=True)
broadcast_prot = BroadcastMetricGatheringUtilities(broadcast_prot_base, user_logger=user_logger)

# *****************

model = ResNetPLModel(num_classes=10, resnet_version='resnet18', lr=0.005,)
model.load_state_dict(torch.load('data/resnet18_svhn.pth', map_location='cpu'))

# *****************
sim = FLSimulator(
    pl_model=model, num_agents=worker_count, communication_rounds=50, client_epochs_per_round=10,
    batch_size=batch_size, dataset_train=dataset[0], dataset_test=dataset[1],
    aggregation_method='fedavg', non_iid_sampling=False, user_logger=user_logger)
# ****
sim.run_simulation(broadcast_prot)


round 1/50 --------------------
  - reporting global model metrics
         - train> loss: 2.18, auc: 0.62, acc: 0.23    |    test> loss: 2.18, auc: 0.61, acc: 0.23

  - setting local models...



/FARM/smohamma/venv/lib/python3.11/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python3 /FARM/smohamma/venv/lib/python3.11/site-packages/ip ...
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


      > training agent 1/2
             + initiating broadcast
         - train> loss: 1.81, auc: 0.81, acc: 0.42    |    test> loss: 2.17, auc: 0.63, acc: 0.24
      > training agent 2/2


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


             + initiating broadcast


...

         - train> loss: 0.54, auc: 1.00, acc: 0.93    |    test> loss: 1.94, auc: 0.75, acc: 0.34
Aggregating gradients using FedAvg...

round 8/50 --------------------
  - reporting global model metrics
         - train> loss: 1.03, auc: 0.98, acc: 0.77    |    test> loss: 1.85, auc: 0.77, acc: 0.38

  - setting local models...



LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


      > training agent 1/2
             + initiating broadcast


Process ForkPoolWorker-2079:
Process ForkPoolWorker-2083:
Exception ignored in: <function _releaseLock at 0x7f9d48956fc0>
Traceback (most recent call last):
  File "/usr/lib/python3.11/logging/__init__.py", line 237, in _releaseLock
    def _releaseLock():
    
KeyboardInterrupt: 
Process ForkPoolWorker-2084:
Process ForkPoolWorker-2072:
Process ForkPoolWorker-2077:
Process ForkPoolWorker-2071:
Process ForkPoolWorker-2080:
Process ForkPoolWorker-2074:
Process ForkPoolWorker-2075:
Process ForkPoolWorker-2076:
Process ForkPoolWorker-2078:
Process ForkPoolWorker-2073:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/lib/python3.11/multiprocessing

KeyboardInterrupt: 